<a href="https://colab.research.google.com/github/NYChase/IBMDataScienceProjects/blob/Capstone-Project/Module_3_Lab_Interactive_Dashboard_with_Plotly_Dash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hands-on Lab: Build an Interactive Dashboard with Plotly Dash

* TASK 1: Add a Launch Site Drop-down Input Component
* TASK 2: Add a callback function to render `success-pie-chart` based on selected site dropdown
* TASK 3: Add a Range Slider to Select Payload
* TASK 4: Add a callback function to render the `success-payload-scatter-chart` scatter plot

Note:Please take screenshots of the Dashboard and save them. Further upload your notebook to github.

The github url and the screenshots are later required in the presentation slides.

After visual analysis using the dashboard, you should be able to obtain some insights to answer the following five questions:

* Which site has the largest successful launches?
* Which site has the highest launch success rate?
* Which payload range(s) has the highest launch success rate?
* Which payload range(s) has the lowest launch success rate?
* Which F9 Booster version (v1.0, v1.1, FT, B4, B5, etc.) has the highest
launch success rate?

In [1]:
!pip install jupyter-dash plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.2 MB/s eta 0:00:00


In [2]:
from jupyter_dash import JupyterDash
from dash import dcc, html
import plotly.express as px

## Download a skeleton dashboard application and dataset
First, let’s get the SpaceX Launch dataset for this lab:

Run the following wget command line in the terminal to download dataset as spacex_launch_dash.csv

In [4]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"

--2025-08-28 04:56:42--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2476 (2.4K) [text/csv]
Saving to: ‘spacex_launch_dash.csv’

spacex_launch_dash. 100%[===================>]   2.42K  --.-KB/s    in 0s      

2025-08-28 04:56:42 (625 MB/s) - ‘spacex_launch_dash.csv’ saved [2476/2476]



Download a skeleton Dash app to be completed in this lab:

In [5]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/t4-Vy4iOU19i8y6E3Px_ww/spacex-dash-app.py"

--2025-08-28 04:56:56--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/t4-Vy4iOU19i8y6E3Px_ww/spacex-dash-app.py
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2075 (2.0K) [application/x-python]
Saving to: ‘spacex-dash-app.py’

spacex-dash-app.py  100%[===================>]   2.03K  --.-KB/s    in 0s      

2025-08-28 04:56:57 (566 MB/s) - ‘spacex-dash-app.py’ saved [2075/2075]



Test the skeleton app by running the following command in the terminal:

## TASK 1: Add a Launch Site Drop-down Input Component


In [19]:
# Import required libraries
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Read the spacex data into pandas dataframe
spacex_df =  pd.read_csv('spacex_launch_dash.csv')
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create a dash application
app = dash.Dash(__name__)

# Create an options list for the dropdown based on unique launch sites
launch_sites = spacex_df['Launch Site'].unique().tolist()
dropdown_options = [{'label': 'All Sites', 'value': 'ALL'}] + [{'label': site, 'value': site} for site in launch_sites]

# Build dash app layout
app.layout = html.Div(children=[html.H1('SpaceX Launch Records Dashboard',
                                        style={'textAlign': 'center', 'color': '#503D36',
                                               'font-size': 40}),
                                # TASK 1: Add a Launch Site Drop-down Input Component
                                html.Div([
                                    html.Label("Select Launch Site:"),
                                    dcc.Dropdown(id='site-dropdown',
                                                 options=dropdown_options,
                                                 value='ALL',
                                                 placeholder="Select a Launch Site here",
                                                 searchable=True
                                                ),
                                    html.Br(),

                                    # TASK 2: Add a pie chart to show the total success launches count for all sites
                                    # If a specific launch site was selected, show the Success vs. Failed counts for the site
                                    html.Div(dcc.Graph(id='success-pie-chart')),
                                    html.Br(),

                                    html.P("Payload range (Kg):"),
                                    # TASK 3: Add a slider to select payload range
                                    dcc.RangeSlider(id='payload-slider',
                                                    min=0, max=10000, step=1000,
                                                    marks={0: '0', 2500: '2500', 5000: '5000',
                                                           7500: '7500', 10000: '10000'},
                                                    value=[min_payload, max_payload]),
                                    html.Br(),

                                    # TASK 4: Add a scatter chart to show the relationship between payload and launch success
                                    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
                                ]),
])

# TASK 2: Add a callback function for `site-dropdown` as input, `success-pie-chart` as output
@app.callback(Output(component_id='success-pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(spacex_df, values='class',
                     names='Launch Site',
                     title='Total Success Launches By Site')
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        fig = px.pie(filtered_df, names='class',
                     title=f'Total Success Launches for site {entered_site}',
                     category_orders={'class': [0, 1]},
                     labels={0: 'Failure', 1: 'Success'})
    return fig

# TASK 4: Add a callback function for `site-dropdown` and `payload-slider` as inputs, `success-payload-scatter-chart` as output
@app.callback(Output(component_id='success-payload-scatter-chart', component_property='figure'),
              [Input(component_id='site-dropdown', component_property='value'),
               Input(component_id='payload-slider', component_property='value')])
def get_scatter_chart(entered_site, payload_range):
    # Filter data based on selected payload range
    filtered_payload_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= payload_range[0]) &
                                     (spacex_df['Payload Mass (kg)'] <= payload_range[1])]

    if entered_site == 'ALL':
        fig = px.scatter(filtered_payload_df, x='Payload Mass (kg)', y='class',
                         color='Booster Version Category',
                         title='Correlation between Payload and Success for all Sites')
    else:
        filtered_site_payload_df = filtered_payload_df[filtered_payload_df['Launch Site'] == entered_site]
        fig = px.scatter(filtered_site_payload_df, x='Payload Mass (kg)', y='class',
                         color='Booster Version Category',
                         title=f'Correlation between Payload and Success for site {entered_site}')
    return fig


# Run the app
if __name__ == '__main__':
    app.run(mode='inline')

<IPython.core.display.Javascript object>

# Finding Insights Visually
Now with the dashboard completed, you should be able to use it to analyze SpaceX launch data, and answer the following questions:

1. Which site has the largest successful launches? **KSC LC-39A at class=10**
2. Which site has the highest launch success rate? **KSC LC-39A at class=10**
3. Which payload range(s) has the highest launch success rate? **launches with payloads between approximately 2000 kg and 6000 kg appear to have a higher concentration of successful outcomes (class=1).**
4. Which payload range(s) has the lowest launch success rate?
5. Which F9 Booster version (`v1.0`, `v1.1`, `FT`, `B4`, `B5`, etc.) has the highest launch success rate? **Booster Version FT and B5 seem to have a higher success rate compared to earlier versions like v1.0 and v1.1.**